In [1]:
import json
import pickle

with open("../config/vectorizer_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

max_len = config["max_len_with_start_and_end_tokens"]

with open("../config/vocab.pkl", "rb") as f:
    vocabulary = pickle.load(f)

In [2]:
import tensorflow as tf
from keras import layers, Model
from keras.applications import InceptionV3

EMBED_DIM = 512
LSTM_UNITS = 512

# =========================================================
# 1. CNN ENCODER (end-to-end, dentro il modello)
# =========================================================
cnn_base = InceptionV3(include_top=False, pooling='avg')
cnn_base.trainable = False  # oppure True se fine-tuning

img_input = layers.Input(shape=(299, 299, 3), name="image")

img_features = cnn_base(img_input)
img_features = layers.Dense(EMBED_DIM, activation='relu')(img_features)

# =========================================================
# 2. CAPTION INPUT (teacher forcing)
# =========================================================
seq_input = layers.Input(shape=(max_len,), name="caption")   # teacher forcing -> max len - 1

word_emb = layers.Embedding(
    input_dim=vocabulary.size + 1,  # mask_zero=True -> input_dim=vocabulary.size + 1
    output_dim=EMBED_DIM,
    mask_zero=True
)(seq_input)

# =========================================================
# 3. INITIAL STATE FROM IMAGE
# =========================================================
h0 = layers.Dense(LSTM_UNITS, activation='tanh')(img_features)
c0 = layers.Dense(LSTM_UNITS, activation='tanh')(img_features)

# =========================================================
# 4. LSTM DECODER
# =========================================================
lstm = layers.LSTM(
    LSTM_UNITS,
    return_sequences=True
)

x = lstm(word_emb, initial_state=[h0, c0])

# =========================================================
# 5. OUTPUT VOCAB DISTRIBUTION
# =========================================================
outputs = layers.Dense(vocabulary.size, activation="softmax")(x)

# =========================================================
# 6. MODEL
# =========================================================
model = Model(inputs=[img_input, seq_input], outputs=outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 299, 299,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inception_v3        │ (None, 2048)      │ 21,802,784 │ image[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption             │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,049,088 │ inception_v3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 512)   │  1,544,704 │ caption[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    262,656 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    262,656 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 20, 512)   │  2,099,200 │ embedding[0][0],  │
│                     │                   │            │ dense_1[0][0],    │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 20, 3016)  │  1,547,208 │ lstm[0][0]        │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 28,568,296 (108.98 MB)

 Trainable params: 6,765,512 (25.81 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [3]:
def loss_ignore_padding(y_true, y_pred):
    # cross entropy per timestep
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true, y_pred
    )  # (batch, time)

    # mask padding
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)  # 0 = PAD_ID

    # applica mask
    loss = loss * mask

    # media normalizzata
    return tf.reduce_sum(loss) / (tf.reduce_sum(mask) + 1e-8)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=loss_ignore_padding
)